# **Project 2: Exploring Data for Impact**
## **What Factors Are Linked to Injury Risk in University Football Players?**

**Student Names:** [Add both names here]  
**Course:** DSC 101  
**Instructor:** Dr. Rimal  
**Date:** [Add submission date]

**Dataset:** University Football Injury Prediction Dataset  
**Primary Source:** https://www.kaggle.com/datasets/yuanchunhong/university-football-injury-prediction-dataset  
**Context Source:** Ma, J., Liu, S., & Pei, Y. (2025). *SHAP-based interpretable machine learning for injury risk prediction in university football players: a multi-dimensional data analysis approach.* Scientific Reports. https://www.nature.com/articles/s41598-025-24144-y

**Working Summary:** The dataset is described as containing data for 800 Chinese university football players and an injury outcome variable for the following season. The published paper above reports that the dataset includes 18 feature variables covering basic information, training factors, physical fitness, and lifestyle habits.

## **How To Use This Template**

1. Download the Kaggle dataset and place the file in a local `data/` folder.
2. Update `DATA_PATH` in the next code cell if your filename is different.
3. Run the notebook once and compare the actual column names with the `standard_column_lookup` dictionary.
4. Replace every placeholder note that starts with `Write about...` before submission.
5. Use the slide template file in this folder to build your presentation.

In [ ]:
# Import the libraries required for data handling, preprocessing, and visualization.
from pathlib import Path
import re

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

# **Introduction**

## **Dataset Background**

Write about where the dataset came from, what population it represents, and why sports injury analysis matters. You can state that the data was obtained from Kaggle and that a 2025 Scientific Reports paper describes the dataset as university football player data with 800 observations and a next-season injury outcome.

## **Project Goal**

Write about your goal in 2-3 sentences. Example:

The goal of this project is to explore how training load, previous injury history, physical fitness, and lifestyle habits relate to injury risk in university football players. Through data cleaning, preprocessing, exploratory data analysis, and visualization, this project aims to identify which factors appear most strongly associated with injuries and how those insights could support prevention strategies.

In [ ]:
# Set the dataset path and load the football injury file.
DATA_PATH = Path("data/university_football_injury_prediction.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Download the Kaggle file and update DATA_PATH if needed."
    )

if DATA_PATH.suffix.lower() == ".csv":
    df_raw = pd.read_csv(DATA_PATH)
elif DATA_PATH.suffix.lower() in {".xlsx", ".xls"}:
    df_raw = pd.read_excel(DATA_PATH)
else:
    raise ValueError("Update DATA_PATH so it points to a CSV or Excel file.")

df_raw.head()

In [ ]:
# Standardize the column names so the rest of the notebook is easier to reuse.
df = df_raw.copy()

def normalize_name(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(name).lower())

standard_column_lookup = {
    "age": "Age",
    "heightcm": "Height_cm",
    "weightkg": "Weight_kg",
    "bmi": "BMI",
    "playingposition": "Playing_Position",
    "traininghoursperweek": "Training_Hours_Per_Week",
    "matchesplayedpastseason": "Matches_Played_Past_Season",
    "previousinjurycount": "Previous_Injury_Count",
    "kneestrengthscore": "Knee_Strength_Score",
    "hamstringflexibility": "Hamstring_Flexibility",
    "reactiontimems": "Reaction_Time_ms",
    "balancetestscore": "Balance_Test_Score",
    "10msprintspeeds": "Sprint_10m_Time_s",
    "10msprinttimes": "Sprint_10m_Time_s",
    "sprint10mtimes": "Sprint_10m_Time_s",
    "agilityscore": "Agility_Score",
    "sleephourspernight": "Sleep_Hours_Per_Night",
    "stresslevelscore": "Stress_Level_Score",
    "nutritionqualityscore": "Nutrition_Quality_Score",
    "warmuproutineadherence": "Warmup_Routine_Adherence",
    "injurynextseason": "Injury_Next_Season"
}

rename_map = {}
for column in df.columns:
    normalized_column = normalize_name(column)
    if normalized_column in standard_column_lookup:
        rename_map[column] = standard_column_lookup[normalized_column]

df = df.rename(columns=rename_map)

expected_columns = [
    "Age",
    "Height_cm",
    "Weight_kg",
    "BMI",
    "Playing_Position",
    "Training_Hours_Per_Week",
    "Matches_Played_Past_Season",
    "Previous_Injury_Count",
    "Knee_Strength_Score",
    "Hamstring_Flexibility",
    "Reaction_Time_ms",
    "Balance_Test_Score",
    "Sprint_10m_Time_s",
    "Agility_Score",
    "Sleep_Hours_Per_Night",
    "Stress_Level_Score",
    "Nutrition_Quality_Score",
    "Warmup_Routine_Adherence",
    "Injury_Next_Season"
]

print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
display(pd.DataFrame({"column": df.columns, "dtype": df.dtypes.astype(str)}))

missing_expected = [column for column in expected_columns if column not in df.columns]
if missing_expected:
    print("Review these columns and update the lookup dictionary if your Kaggle file uses different names:")
    print(missing_expected)

## **Variables To Describe In Your Write-Up**

Use this section to explain the meaning of the main variables after you confirm the exact column names in your file.

- **Target variable:** `Injury_Next_Season` (0 = not injured, 1 = injured)
- **Basic information variables:** age, height, weight, BMI, playing position
- **Training-related variables:** training hours per week, matches played in the past season, previous injury count
- **Physical fitness variables:** knee strength, hamstring flexibility, reaction time, balance score, 10m sprint time, agility score
- **Lifestyle variables:** sleep hours, stress level, nutrition quality, warmup routine adherence

Write about the number of rows, the number of variables, and the data types shown above.

# **Data Cleaning And Preprocessing**

Write about the cleaning plan before the code output. Example: first check duplicates and missing values, then standardize data types, convert categorical variables into numerical form, and inspect outliers before choosing how to handle them.

In [ ]:
# Clean duplicates, convert data types, and prepare the target and categorical variables.
df_clean = df.copy()
rows_before = len(df_clean)
df_clean = df_clean.drop_duplicates()
rows_after = len(df_clean)

numeric_columns = [
    "Age",
    "Height_cm",
    "Weight_kg",
    "BMI",
    "Training_Hours_Per_Week",
    "Matches_Played_Past_Season",
    "Previous_Injury_Count",
    "Knee_Strength_Score",
    "Hamstring_Flexibility",
    "Reaction_Time_ms",
    "Balance_Test_Score",
    "Sprint_10m_Time_s",
    "Agility_Score",
    "Sleep_Hours_Per_Night",
    "Stress_Level_Score",
    "Nutrition_Quality_Score",
    "Warmup_Routine_Adherence"
]

for column in numeric_columns:
    if column in df_clean.columns:
        df_clean[column] = pd.to_numeric(df_clean[column], errors="coerce")

if "Injury_Next_Season" in df_clean.columns:
    injury_map = {
        "0": 0,
        "1": 1,
        "no": 0,
        "yes": 1,
        "not injured": 0,
        "injured": 1,
        "non-injured": 0,
        "non injured": 0
    }
    if not pd.api.types.is_numeric_dtype(df_clean["Injury_Next_Season"]):
        df_clean["Injury_Next_Season"] = (
            df_clean["Injury_Next_Season"]
            .astype(str)
            .str.strip()
            .str.lower()
            .map(injury_map)
        )
    df_clean["Injury_Next_Season"] = pd.to_numeric(df_clean["Injury_Next_Season"], errors="coerce").astype("Int64")

if "Playing_Position" in df_clean.columns:
    df_clean["Playing_Position"] = df_clean["Playing_Position"].astype("category")
    df_clean["Playing_Position_Code"] = df_clean["Playing_Position"].cat.codes

missing_summary = (
    df_clean.isna()
    .sum()
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "column"})
)
missing_summary["missing_percent"] = 100 * missing_summary["missing_count"] / len(df_clean)

print(f"Duplicate rows removed: {rows_before - rows_after}")
display(missing_summary.sort_values("missing_count", ascending=False))
df_clean.head()

In [ ]:
# Detect outliers with the IQR rule and cap extreme values for the analysis dataset.
analysis_numeric_columns = [column for column in numeric_columns if column in df_clean.columns]

def build_iqr_summary(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    summary_rows = []
    for column in columns:
        q1 = frame[column].quantile(0.25)
        q3 = frame[column].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        outlier_mask = (frame[column] < lower_bound) | (frame[column] > upper_bound)
        summary_rows.append(
            {
                "column": column,
                "lower_bound": lower_bound,
                "upper_bound": upper_bound,
                "outlier_count": int(outlier_mask.sum()),
                "outlier_percent": round(100 * outlier_mask.mean(), 2)
            }
        )
    return pd.DataFrame(summary_rows)

outlier_summary_before = build_iqr_summary(df_clean, analysis_numeric_columns)
display(outlier_summary_before.sort_values("outlier_count", ascending=False))

df_model = df_clean.copy()
for column in analysis_numeric_columns:
    q1 = df_model[column].quantile(0.25)
    q3 = df_model[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    df_model[column] = df_model[column].clip(lower=lower_bound, upper=upper_bound)

outlier_summary_after = build_iqr_summary(df_model, analysis_numeric_columns)
display(outlier_summary_after.sort_values("outlier_count", ascending=False))

Write about the cleaning results here. Mention whether there were missing values, whether duplicates were removed, how categorical data was encoded, and whether outliers were capped with the IQR rule. If the dataset was already mostly clean, say that clearly.

# **Exploratory Data Analysis (EDA)**

Use the following plots to explain distributions, compare injured versus non-injured players, and identify patterns that matter for injury risk.

In [ ]:
# Calculate descriptive statistics and visualize class balance and position-level injury rates.
display(df_model[analysis_numeric_columns].describe().T)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

if "Injury_Next_Season" in df_model.columns:
    sns.countplot(data=df_model, x="Injury_Next_Season", ax=axes[0])
    axes[0].set_title("Injury Outcome Distribution")
    axes[0].set_xlabel("Injury_Next_Season")
    axes[0].set_ylabel("Player Count")

if {"Playing_Position", "Injury_Next_Season"}.issubset(df_model.columns):
    position_rate = (
        df_model.groupby("Playing_Position", observed=True)["Injury_Next_Season"]
        .mean()
        .sort_values(ascending=False)
        .reset_index(name="injury_rate")
    )
    sns.barplot(data=position_rate, x="Playing_Position", y="injury_rate", ax=axes[1])
    axes[1].set_title("Average Injury Rate by Playing Position")
    axes[1].set_xlabel("Playing Position")
    axes[1].set_ylabel("Injury Rate")
    axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
# Plot feature distributions for key variables that may be linked to injury risk.
distribution_columns = [
    column for column in [
        "Training_Hours_Per_Week",
        "Previous_Injury_Count",
        "Sleep_Hours_Per_Night",
        "Stress_Level_Score",
        "Balance_Test_Score",
        "Sprint_10m_Time_s"
    ] if column in df_model.columns
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for axis, column in zip(axes, distribution_columns):
    sns.histplot(data=df_model, x=column, kde=True, ax=axis)
    axis.set_title(f"Distribution of {column}")

for axis in axes[len(distribution_columns):]:
    axis.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Compare key numeric variables across injury outcomes using boxplots.
comparison_columns = [
    column for column in [
        "Stress_Level_Score",
        "Sleep_Hours_Per_Night",
        "Previous_Injury_Count",
        "Balance_Test_Score",
        "Training_Hours_Per_Week",
        "Warmup_Routine_Adherence"
    ] if column in df_model.columns
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for axis, column in zip(axes, comparison_columns):
    sns.boxplot(data=df_model, x="Injury_Next_Season", y=column, ax=axis)
    axis.set_title(f"{column} by Injury Outcome")

for axis in axes[len(comparison_columns):]:
    axis.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Visualize correlations and group means to identify the strongest injury-related patterns.
correlation_columns = analysis_numeric_columns.copy()
if "Playing_Position_Code" in df_model.columns:
    correlation_columns.append("Playing_Position_Code")
if "Injury_Next_Season" in df_model.columns:
    correlation_columns.append("Injury_Next_Season")

correlation_matrix = df_model[correlation_columns].corr(numeric_only=True)

plt.figure(figsize=(14, 10))
sns.heatmap(correlation_matrix, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

group_mean_columns = [
    column for column in [
        "Training_Hours_Per_Week",
        "Previous_Injury_Count",
        "Sleep_Hours_Per_Night",
        "Stress_Level_Score",
        "Balance_Test_Score",
        "Warmup_Routine_Adherence"
    ] if column in df_model.columns
]

if "Injury_Next_Season" in df_model.columns:
    group_means = df_model.groupby("Injury_Next_Season")[group_mean_columns].mean().T
    display(group_means)

In [ ]:
# Use scatterplots to examine relationships between multiple risk factors at the same time.
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

if {"Stress_Level_Score", "Sleep_Hours_Per_Night", "Injury_Next_Season"}.issubset(df_model.columns):
    sns.scatterplot(
        data=df_model,
        x="Stress_Level_Score",
        y="Sleep_Hours_Per_Night",
        hue="Injury_Next_Season",
        alpha=0.7,
        ax=axes[0]
    )
    axes[0].set_title("Stress vs Sleep by Injury Outcome")

if {"Training_Hours_Per_Week", "Previous_Injury_Count", "Injury_Next_Season"}.issubset(df_model.columns):
    sns.scatterplot(
        data=df_model,
        x="Training_Hours_Per_Week",
        y="Previous_Injury_Count",
        hue="Injury_Next_Season",
        alpha=0.7,
        ax=axes[1]
    )
    axes[1].set_title("Training Load vs Previous Injury Count")

plt.tight_layout()
plt.show()

## **Write About Your EDA Findings**

Write 1-3 short paragraphs here about the main patterns from the tables and plots above.

Useful points to cover:
- Was the injury outcome balanced or imbalanced?
- Which variables looked most different between injured and non-injured players?
- Did higher stress, lower sleep, or higher previous injury counts appear to matter?
- Did certain playing positions show higher average injury rates?
- Which variables seemed strongly correlated with injury outcome?

# **Visualization Discussion**

For the assignment, explain each graph briefly. A simple approach is to write 2-3 sentences under each major figure in your final version. Mention what the chart shows, what pattern stands out, and why the pattern matters for injury prevention or athlete health.

# **Summary Of Findings**

Write at least 500 words in this section before submission.

Checklist for your final summary:
- Summarize the most important EDA and visualization insights.
- Explain the value of the project to sports programs, coaches, trainers, or society.
- Describe next steps such as adding more seasons of data, validating findings, or comparing models.
- Discuss ethical considerations, including fairness, privacy, over-reliance on prediction tools, and careful use of athlete health data.
- Mention important limitations, such as the dataset source, sample size, or limited generalizability.

Suggested summary starter:

This project explored the relationship between training habits, physical fitness, lifestyle factors, and injury risk in university football players. After cleaning and preprocessing the dataset, the exploratory analysis suggested that several modifiable factors may be associated with injury outcomes. In particular, variables such as stress level, sleep duration, previous injury history, and physical fitness measures appeared to show meaningful differences between injured and non-injured players...

# **References**

1. Kaggle dataset page: https://www.kaggle.com/datasets/yuanchunhong/university-football-injury-prediction-dataset
2. Ma, J., Liu, S., & Pei, Y. (2025). *SHAP-based interpretable machine learning for injury risk prediction in university football players: a multi-dimensional data analysis approach.* Scientific Reports. https://www.nature.com/articles/s41598-025-24144-y

Add any additional sources you use for background or sports injury context.